# ResNet-18 on a Kaggle GPU

This notebook is a **GPU-only fork of `kaggle_comp.ipynb`**. Its only job is to run section 3.9 of the main notebook: train the pretrained ResNet-18 on a Kaggle GPU and write its submission file. The split, transforms, dataset classes, training functions and ResNet-18 cells are copied from `kaggle_comp.ipynb`, so they behave exactly as they would in the main notebook.

**Image size experiment:** `RESNET_IMAGE_SIZE` in the settings cell is the only thing to change between runs. Every run writes the same file names (`resnet-18_history.csv`, `resnet-18_best.pt`, `submission_resnet-18.csv`, ...), so download each run's output into its own folder and rename the files by hand, as in the `resnet-18_results_run*` folders. The data analysis and the baseline CNN stay in the main notebook. The main notebook (`kaggle_comp.ipynb`, one folder up) runs on the laptop only, and this notebook copies its pipeline functions (the dataset classes, training, evaluation and prediction functions and the ResNet-18 helpers) for GPU training, so a change to those functions must be made in both notebooks.

**The private Kaggle dataset (`kaggle-comp-files`) must contain:**
- `images_info.csv`: the image table from the main notebook's image analysis (used here only to list the test images).
- `train_split.csv` and `validation_split.csv`: the split from section 2.2, so the validation images are identical to the laptop's.
- `competition_data.dat` (optional): the competition folder zipped and renamed to `.dat`, so Kaggle keeps it as one file instead of unzipping it. If it is not in the dataset, attach the competition data as an input instead.

**How to run it on Kaggle:**
1. Add the inputs: the `kaggle-comp-files` dataset (plus the competition data, if `competition_data.dat` is not in the dataset).
2. Session options: **Accelerator → GPU** (T4 x2 or P100) and **Internet → On** (the pretrained ResNet-18 weights are downloaded).
3. Check that `SMOKE_TEST = False` in the run settings cell.
4. **Save Version → Save & Run All (Commit)**. When the run finishes, download the files listed by the last cell from the Output tab.

**Testing on the laptop:** set `SMOKE_TEST = True` to run every cell on a small sample in a few minutes on the CPU. Laptop outputs go to `resnet18_test_output/`, so a test run never overwrites the main notebook's `figures/`, `models/` or `submissions/`.

### Imports

Every library used in this notebook.

In [ ]:
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

### Workplace Check

`SMOKE_TEST = True` runs the whole notebook on a small sample (640 training and 320 validation images, 1 epoch per stage, 640 test images), to check it works before the full run. It must be `False` for the real Kaggle run.

In [ ]:
SMOKE_TEST = False                                 # default -> True = quick test on a small sample (laptop CPU); False = the full Kaggle run

# Kaggle folders
KAGGLE_INPUT_FOLDER = Path("/kaggle/input")        # Attached datasets and competition data (read-only)
KAGGLE_WORKING_FOLDER = Path("/kaggle/working")    # Everything written here is saved as the run's output
KAGGLE_TEMP_FOLDER = Path("/kaggle/temp")          # Scratch space for the unpacked competition images (not saved as output)

RUNNING_ON_KAGGLE = KAGGLE_INPUT_FOLDER.exists()   # /kaggle/input only exists on Kaggle
if RUNNING_ON_KAGGLE: SMOKE_TEST = False           # If running on Kaggle, do the full run (not a smoke test)

print("Running on Kaggle:", RUNNING_ON_KAGGLE)
print("Smoke test:       ", SMOKE_TEST)

### GPU check

Training ResNet-18 on a CPU would take many hours, so on Kaggle the notebook stops here with an error if no GPU is turned on. On the laptop it carries on with the CPU, for smoke tests.

In [ ]:
print("PyTorch version:    ", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("GPU available:      ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:           ", torch.cuda.get_device_name(0))   # e.g. Tesla T4 or Tesla P100

if RUNNING_ON_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Turn on a GPU accelerator in Session options (Accelerator: GPU T4 x2 or GPU P100), then run the notebook again.")

### Finding the input folders

Kaggle's folder names under `/kaggle/input` depend on how the data was attached, so instead of typing them, `find_folder_containing` searches every folder for a file that is known to be there (for example `images_info.csv`) and returns the folder it is in.

In [ ]:
def find_folder_containing(search_folder, file_name):
    """Return the folder (anywhere inside search_folder) that contains file_name. Stops with an error if it is not found."""
    matches = []                                                            # Every folder that contains the file
    for folder_path, subfolder_names, file_names in os.walk(search_folder, followlinks=True):   # Visit every folder and subfolder
        if file_name in file_names:
            matches.append(Path(folder_path))

    if len(matches) == 0:
        raise FileNotFoundError(f"Could not find {file_name} anywhere in {search_folder}. Check the inputs attached to the notebook.")
    if len(matches) > 1:
        print(f"Warning: {file_name} was found in {len(matches)} folders, using the first: {matches[0]}")
    return matches[0]

### Data paths

**On Kaggle:** `my_dataset_folder` is the folder that contains `images_info.csv`. If it also contains `competition_data.dat`, that zip file is unpacked once into `/kaggle/temp/competition_data` and used as the competition data; otherwise the attached competition data is found by looking for `train_labels.csv`. Outputs go to `/kaggle/working`, so they are saved with the run.

The competition download keeps the images one level deeper than expected (`train/train` and `test/test`), so the image folders are checked rather than assumed.

In [ ]:
if RUNNING_ON_KAGGLE:
    my_dataset_folder = find_folder_containing(KAGGLE_INPUT_FOLDER, "images_info.csv")   # My private dataset of files made on the laptop

    competition_archive_path = my_dataset_folder / "competition_data.dat"           # The competition folder as a zip file (optional)
    if competition_archive_path.exists():
        unpacked_folder = KAGGLE_TEMP_FOLDER / "competition_data"                   # Where the zip file is unpacked (not saved as output)
        unpack_done_marker = unpacked_folder / "UNPACK_COMPLETE"                     # Written after a successful unpack, so it only happens once
        if unpack_done_marker.exists():
            print("Competition data already unpacked in", unpacked_folder)
        else:
            print("Unpacking", competition_archive_path, "into", unpacked_folder, "...")
            start_time = time.perf_counter()
            unpacked_folder.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(competition_archive_path, unpacked_folder, format="zip")   # The .dat file is really a zip file
            unpack_done_marker.touch()                                               # Mark the unpack as finished
            print(f"Unpacked in {(time.perf_counter() - start_time) / 60:.1f} minutes")
        data_folder = find_folder_containing(unpacked_folder, "train_labels.csv")
    else:
        data_folder = find_folder_containing(KAGGLE_INPUT_FOLDER, "train_labels.csv")    # Competition data attached as an input

    images_info_csv_path = my_dataset_folder / "images_info.csv"
    train_split_csv_path = my_dataset_folder / "train_split.csv"
    validation_split_csv_path = my_dataset_folder / "validation_split.csv"
    output_folder = KAGGLE_WORKING_FOLDER                                            # Saved as the run's output

# The images are in train/train and test/test in the competition download; use the single level if that is how they were stored
if (data_folder / "train" / "train").exists():
    training_images_folder = data_folder / "train" / "train"
else:
    training_images_folder = data_folder / "train"
if (data_folder / "test" / "test").exists():
    testing_images_folder = data_folder / "test" / "test"
else:
    testing_images_folder = data_folder / "test"

sample_submission_csv_path = data_folder / "sample_submission.csv"          # Sample submission CSV file path
train_labels_csv_path = data_folder / "train_labels.csv"                    # Training labels CSV file path

# Print every folder and file used, and whether it exists
paths_to_check = {
    "My dataset folder": my_dataset_folder,
    "Competition data folder": data_folder,
    "Training images": training_images_folder,
    "Test images": testing_images_folder,
    "images_info.csv": images_info_csv_path,
    "train_split.csv": train_split_csv_path,
    "validation_split.csv": validation_split_csv_path,
    "train_labels.csv": train_labels_csv_path,
    "sample_submission.csv": sample_submission_csv_path,
    "Output folder": output_folder,
}
for name, path in paths_to_check.items():
    print(f"{name:<24} exists: {str(path.exists()):<5}  {path}")
print()

df_sample_submission = pd.read_csv(sample_submission_csv_path)  # Load sample submission CSV into a DataFrame
df_train_labels = pd.read_csv(train_labels_csv_path)            # Load training labels CSV into a DataFrame
print("Rows in train_labels.csv:     ", len(df_train_labels), "(should be 63,660)")
print("Rows in sample_submission.csv:", len(df_sample_submission), "(should be 29,443)")
print("Training images found:        ", len(list(training_images_folder.glob("*.jpg"))))
print("Test images found:            ", len(list(testing_images_folder.glob("*.jpg"))))

### Settings

The same values as `kaggle_comp.ipynb`: the seed and classes (Task 1), batch size, ImageNet normalisation, seeding and device (Task 2), and the ResNet-18 settings (section 3.8). `USE_AUGMENTATION` switches the random augmentation of the training images on (the default, as in Task 2) or off, for the augmentation experiment. The output folders are created inside `output_folder`.

In [ ]:
# Global settings
RANDOM_SEED = 42                                  # Same seed = same random sample images every run
CLASSES = ["CatEye", "CatFace", "CatMouth", "CatNose", "DogEye", "DogFace", "DogMouth", "DogNose"]
BATCH_SIZE = 64                                   # Number of images the model sees in each training step

# Mean and standard deviation of the red, green and blue channels of the ImageNet dataset.
# These are fixed, published ImageNet values, so nothing is calculated from the validation or test data (no data leakage).
# Every model uses them, and pretrained torchvision models expect images normalised with these values.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Fix every random number generator, so the split, augmentation and training are the same every run
random.seed(RANDOM_SEED)                          # Python's own random module
np.random.seed(RANDOM_SEED)                       # NumPy
torch.manual_seed(RANDOM_SEED)                    # PyTorch

# Use a GPU if there is one, otherwise the CPU
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")                 # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")                  # Apple Silicon GPU
else:
    DEVICE = torch.device("cpu")

# ResNet-18 settings (the same as section 3.8)
RESNET_IMAGE_SIZE = 128                           # ResNet-18 input size: the only setting to change between runs (128 or 224; 224 is about 3 times slower)
RESNET_FROZEN_EPOCHS = 3                          # Stage 1: train only the new fc layer, backbone frozen
RESNET_FROZEN_LEARNING_RATE = 0.001               # Adam's default: the new layer starts from random weights
RESNET_FINE_TUNE_EPOCHS = 7                       # Stage 2: train the whole network
RESNET_FINE_TUNE_LEARNING_RATE = 0.0001           # 10 times smaller, so the pretrained weights are adjusted gently, not overwritten
USE_AUGMENTATION = True                           # Default True = randomly augment the training images (Task 2); False = resize and normalise only, like validation
KAGGLE_NUM_WORKERS = 4                            # Kaggle has 4 CPU cores; reduce to 2 or 0 if the DataLoader raises worker or shared-memory errors

# DataLoader worker processes
if RUNNING_ON_KAGGLE:
    NUM_WORKERS = KAGGLE_NUM_WORKERS              # Extra processes prepare the next batches while the GPU trains
else:
    NUM_WORKERS = 0                               # Laptop (macOS): worker processes cannot load Dataset classes defined in a notebook

# Output folders (inside /kaggle/working on Kaggle, so they are saved with the run)
figures_folder = output_folder / "figures"        # Learning curves and history
models_folder = output_folder / "models"          # Best weights
submissions_folder = output_folder / "submissions"   # Kaggle submission file
figures_folder.mkdir(parents=True, exist_ok=True) # Create the folders if they don't exist yet
models_folder.mkdir(parents=True, exist_ok=True)
submissions_folder.mkdir(parents=True, exist_ok=True)

print("Device:             ", DEVICE)
print("Image size:         ", RESNET_IMAGE_SIZE, "x", RESNET_IMAGE_SIZE)
print("Batch size:         ", BATCH_SIZE)
print("Stage 1 (frozen):   ", RESNET_FROZEN_EPOCHS, "epochs, learning rate", RESNET_FROZEN_LEARNING_RATE)
print("Stage 2 (fine-tune):", RESNET_FINE_TUNE_EPOCHS, "epochs, learning rate", RESNET_FINE_TUNE_LEARNING_RATE)
print("Augmentation:       ", USE_AUGMENTATION)
print("DataLoader workers: ", NUM_WORKERS)
print("Output folders:     ", figures_folder, "|", models_folder, "|", submissions_folder)

### Data

`images_info.csv` is loaded only to get the list of test images (`df_test_info`). The training and validation splits are loaded from the CSV files saved in section 2.2. This is never made again, so the validation images are exactly the same as on the laptop.

In [ ]:
df_info = pd.read_csv(images_info_csv_path, dtype={"class": "string"})          # "class" is empty for test images
df_test_info = df_info[df_info["split"] == "test"]                               # One row per test image, in image ID order

df_train_split = pd.read_csv(train_split_csv_path, dtype={"class": "string"})    # The training split from section 2.2
df_validation_split = pd.read_csv(validation_split_csv_path, dtype={"class": "string"})   # The validation split from section 2.2

print("Training images:  ", len(df_train_split), "(should be 50,917)")
print("Validation images:", len(df_validation_split), "(should be 12,728)")
print("Test images:      ", len(df_test_info), "(should be 29,443)")

### Encode the class labels as numbers (copied from section 2.4)

The model outputs one score per class, so each class name is given a number from 0 to 7, in the order of `CLASSES`. `index_to_class` turns the model's predicted numbers back into class names for the Kaggle submission.

In [ ]:
class_to_index = {}                                             # Class name -> number, e.g. "CatEye" -> 0
index_to_class = {}                                             # Number -> class name, e.g. 0 -> "CatEye"
for index in range(len(CLASSES)):
    class_name = CLASSES[index]
    class_to_index[class_name] = index
    index_to_class[index] = class_name

print("class_to_index:", class_to_index)
print("index_to_class:", index_to_class)
print()

# Add the number for each image's class as a new "label" column
df_train_split["label"] = df_train_split["class"].map(class_to_index)            # Look up each class name in the dictionary
df_validation_split["label"] = df_validation_split["class"].map(class_to_index)

# Every row should have a label (map() gives an empty value for a class name not in the dictionary)
print("Missing labels in training split:  ", df_train_split["label"].isna().sum())
print("Missing labels in validation split:", df_validation_split["label"].isna().sum())
print()

print("First 5 rows of df_train_split:")
print(df_train_split[["image_id", "file_name", "class", "label"]].head())

### Smoke test sample

When `SMOKE_TEST` is `True`, the data is cut down so the whole notebook runs in a few minutes on a CPU: 640 training and 320 validation images, sampled at random with `RANDOM_SEED` (so all 8 classes are included, not just the first class in the file), 1 epoch per stage, and only the first 640 test images. When it is `False`, nothing changes.

In [ ]:
if SMOKE_TEST:
    df_train_split = df_train_split.sample(640, random_state=RANDOM_SEED)           # 640 random training images (all 8 classes)
    df_validation_split = df_validation_split.sample(320, random_state=RANDOM_SEED) # 320 random validation images
    df_test_info = df_test_info.iloc[0:640]                                          # First 640 test images, to check prediction and the file format
    RESNET_FROZEN_EPOCHS = 1                                                         # 1 epoch per stage
    RESNET_FINE_TUNE_EPOCHS = 1

    print("SMOKE TEST: training", len(df_train_split), "| validation", len(df_validation_split), "| test", len(df_test_info), "| 1 epoch per stage")
    print("Classes in the smoke-test training images:", df_train_split["class"].value_counts().to_dict())
else:
    print("Full run: training", len(df_train_split), "| validation", len(df_validation_split), "| test", len(df_test_info),
          "|", RESNET_FROZEN_EPOCHS, "+", RESNET_FINE_TUNE_EPOCHS, "epochs")

### Image paths (copied from Task 1)

Returns the full path of a training or test image.

In [ ]:
def get_image_path(split, file_name):
    """Return the full path of an image."""
    if split == "train":
        return training_images_folder / file_name
    else:
        return testing_images_folder / file_name

### Dataset classes (copied from section 2.6)

A PyTorch `Dataset` tells the DataLoader how many images there are (`__len__`) and how to load image number `i` (`__getitem__`). Images are opened from disk one at a time when they are needed, so the whole dataset never has to fit in memory.

`FacialFeatureDataset` is for the labelled training and validation images and returns `(image_tensor, label)`.

In [ ]:
class FacialFeatureDataset(Dataset):
    """Labelled images: returns (image_tensor, label) for each row of a dataframe."""

    def __init__(self, df, transform):
        self.df = df                                                    # One row per image (needs split, file_name and label columns)
        self.transform = transform                                      # Transform applied to every image (training or evaluation)

    def __len__(self):
        return len(self.df)                                             # Number of images in the dataset

    def __getitem__(self, i):
        row = self.df.iloc[i]                                           # The i-th row of the dataframe
        image_path = get_image_path(row["split"], row["file_name"])     # Full path of the image file
        image = Image.open(image_path).convert("RGB")                   # Open the image and make sure it has 3 colour channels
        image_tensor = self.transform(image)                            # Resize, (augment) and normalise -> 3 x IMAGE_SIZE x IMAGE_SIZE tensor
        label = int(row["label"])                                       # Class number 0-7 (int, so the DataLoader makes a tensor of whole numbers)
        return image_tensor, label

`TestImageDataset` is for the Kaggle test images. They have no label, so it returns `(image_tensor, image_id)` instead, and the ID is used to match each prediction to its image in the submission file.

In [ ]:
class TestImageDataset(Dataset):
    """Unlabelled test images: returns (image_tensor, image_id) for each row of a dataframe."""

    def __init__(self, df, transform):
        self.df = df                                                    # One row per image (needs split, file_name and image_id columns)
        self.transform = transform                                      # Transform applied to every image (always the evaluation transform)

    def __len__(self):
        return len(self.df)                                             # Number of images in the dataset

    def __getitem__(self, i):
        row = self.df.iloc[i]                                           # The i-th row of the dataframe
        image_path = get_image_path(row["split"], row["file_name"])     # Full path of the image file
        image = Image.open(image_path).convert("RGB")                   # Open the image and make sure it has 3 colour channels
        image_tensor = self.transform(image)                            # Resize and normalise -> 3 x IMAGE_SIZE x IMAGE_SIZE tensor
        image_id = int(row["image_id"])                                 # Image ID, used to write the prediction to the right row of the submission
        return image_tensor, image_id

### Training and evaluation functions (copied from section 3.3)

`train_one_epoch` passes every training batch through the model once and updates the weights after each batch, following the four steps of training a neural network (Lecture 9):
1. **Forward pass:** the images go through the model to get the 8 scores for each image.
2. **Loss:** the scores are compared with the true labels.
3. **Gradients (backpropagation):** `loss.backward()` works out how much each weight contributed to the loss.
4. **Weight update:** `optimiser.step()` moves every weight a small step in the direction that lowers the loss.

The function returns the average loss and the accuracy (share of images whose highest score is the true class) over the whole epoch.

In [ ]:
def train_one_epoch(model, loader, loss_function, optimiser):
    """Train the model on every batch in loader once. Returns (average loss, accuracy)."""
    model.train()                                                   # Training mode: dropout on, batch norm learns from each batch

    total_loss = 0.0                                                # Sum of the loss over every image so far
    total_correct = 0                                               # Number of correct predictions so far
    total_images = 0                                                # Number of images seen so far

    for images, labels in loader:                                   # One batch at a time
        images = images.to(DEVICE)                                  # Move the batch to the same device as the model
        labels = labels.to(DEVICE)

        optimiser.zero_grad()                                       # Clear the gradients from the previous batch (PyTorch adds new gradients to old ones otherwise)
        outputs = model(images)                                     # Lecture 9 step 1, forward pass: images -> 8 scores per image
        loss = loss_function(outputs, labels)                       # Lecture 9 step 2, loss: how far the scores are from the true labels (batch average)
        loss.backward()                                             # Lecture 9 step 3, backpropagation: gradient of the loss for every weight
        optimiser.step()                                            # Lecture 9 step 4, weight update: Adam moves each weight to lower the loss

        batch_size = labels.size(0)                                 # Images in this batch (the last batch can be smaller than BATCH_SIZE)
        total_loss += loss.item() * batch_size                      # Batch average x batch size = batch total, so every image counts equally
        predictions = torch.argmax(outputs, dim=1)                  # Predicted class = index of the biggest score in each row
        total_correct += (predictions == labels).sum().item()       # Count the predictions that match the true label
        total_images += batch_size

    average_loss = total_loss / total_images                        # Average loss per image over the epoch
    accuracy = total_correct / total_images                         # Share of images predicted correctly (0 to 1)
    return average_loss, accuracy

`evaluate` measures the model on images it does not learn from (the validation set). It works out the same loss and accuracy, but only does the forward pass: there is no `backward()` or `step()`, so the weights do not change. `model.eval()` turns dropout off and makes batch norm use the values it learned during training, so the result is the same every time.

In [ ]:
def evaluate(model, loader, loss_function):
    """Measure the model on every batch in loader without changing its weights. Returns (average loss, accuracy)."""
    model.eval()                                                    # Evaluation mode: dropout off, batch norm uses its stored values

    total_loss = 0.0                                                # Sum of the loss over every image so far
    total_correct = 0                                               # Number of correct predictions so far
    total_images = 0                                                # Number of images seen so far

    with torch.no_grad():                                           # No gradients needed (no backpropagation), which saves time and memory
        for images, labels in loader:                               # One batch at a time
            images = images.to(DEVICE)                              # Move the batch to the same device as the model
            labels = labels.to(DEVICE)

            outputs = model(images)                                 # Forward pass only: images -> 8 scores per image
            loss = loss_function(outputs, labels)                   # Loss for this batch (batch average)

            batch_size = labels.size(0)                             # Images in this batch
            total_loss += loss.item() * batch_size                  # Batch average x batch size = batch total
            predictions = torch.argmax(outputs, dim=1)              # Predicted class = index of the biggest score in each row
            total_correct += (predictions == labels).sum().item()   # Count the predictions that match the true label
            total_images += batch_size

    average_loss = total_loss / total_images                        # Average loss per image
    accuracy = total_correct / total_images                         # Share of images predicted correctly (0 to 1)
    return average_loss, accuracy

### Training loop (copied from section 3.4)

`train_model` trains for a set number of epochs. Each epoch it trains on every training batch, then measures the model on the validation set, so overfitting can be spotted (training accuracy keeps rising while validation accuracy stops improving). Whenever the validation accuracy is the best so far, the weights are saved, because accuracy is the competition metric. This means the final model is the best one seen, not just the last one. The loss and accuracy of every epoch are returned as a table, for the learning curves in Task 4.

In [ ]:
def train_model(model, train_loader, validation_loader, loss_function, optimiser, epochs, model_name):
    """Train for a number of epochs, save the weights with the best validation accuracy, and return the history as a dataframe."""
    weights_path = models_folder / f"{model_name}_best.pt"                    # Where the best weights are saved
    best_validation_accuracy = 0.0                                             # Best validation accuracy so far
    history = []                                                               # One dictionary of results per epoch

    for epoch in range(1, epochs + 1):                                         # Epochs are numbered from 1
        start_time = time.perf_counter()                                       # Start the timer for this epoch

        train_loss, train_accuracy = train_one_epoch(model, train_loader, loss_function, optimiser)   # Learn from every training batch once
        validation_loss, validation_accuracy = evaluate(model, validation_loader, loss_function)      # Measure on the validation images (no learning)

        epoch_minutes = (time.perf_counter() - start_time) / 60                # Time taken for this epoch, in minutes

        print(f"Epoch {epoch}/{epochs} | "
              f"train loss {train_loss:.3f}, accuracy {train_accuracy * 100:.1f}% | "
              f"validation loss {validation_loss:.3f}, accuracy {validation_accuracy * 100:.1f}% | "
              f"{epoch_minutes:.1f} min")

        # Record this epoch's results
        epoch_results = {}
        epoch_results["epoch"] = epoch
        epoch_results["train_loss"] = train_loss
        epoch_results["train_accuracy"] = train_accuracy
        epoch_results["validation_loss"] = validation_loss
        epoch_results["validation_accuracy"] = validation_accuracy
        epoch_results["minutes"] = epoch_minutes
        history.append(epoch_results)

        # Save the weights if this is the best validation accuracy so far
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            torch.save(model.state_dict(), weights_path)                       # state_dict() = every weight of the model, by layer name
            print(f"    New best validation accuracy ({validation_accuracy * 100:.1f}%): weights saved to {weights_path}")

    df_history = pd.DataFrame(history)                                         # One row per epoch
    return df_history

### Predictions (copied from section 3.7)

`predict` passes every test batch through the model and keeps the class with the highest score for each image. It uses `model.eval()` and `torch.no_grad()`, so dropout is off and the weights do not change. The test loader is not shuffled, and each batch carries the image IDs, so every prediction stays matched to its image.

In [ ]:
def predict(model, loader):
    """Predict a class number for every image in loader. Returns (list of image IDs, list of predicted class numbers)."""
    model.eval()                                                    # Evaluation mode: dropout off, batch norm uses its stored values

    all_image_ids = []                                              # Image IDs in the order they were predicted
    all_predictions = []                                            # Predicted class number (0-7) for each image

    with torch.no_grad():                                           # No gradients needed, which saves time and memory
        for images, image_ids in loader:                            # The test dataset returns (image, image_id)
            images = images.to(DEVICE)                              # Move the batch to the same device as the model
            outputs = model(images)                                 # Forward pass: 8 scores per image
            predictions = torch.argmax(outputs, dim=1)              # Predicted class = index of the biggest score
            all_image_ids.extend(image_ids.tolist())                # .tolist() turns the tensor into plain Python integers
            all_predictions.extend(predictions.tolist())

    return all_image_ids, all_predictions

`print_check` prints PASS or FAIL for one submission check (copied from section 3.7).

In [ ]:
def print_check(description, passed):
    """Print PASS or FAIL for one check."""
    if passed:
        print("PASS:", description)
    else:
        print("FAIL:", description)

### Loss function (copied from section 3.2)

`CrossEntropyLoss` is the standard loss for choosing one class out of several. It applies softmax to the model's 8 scores and takes minus the log of the probability given to the true class.

In [ ]:
loss_function = nn.CrossEntropyLoss()                                           # Softmax + minus the log of the true class's probability, averaged over the batch

### ResNet-18 helper functions (copied from section 3.8)

`build_resnet18` loads ResNet-18 with its ImageNet weights (downloaded the first time, then loaded from the cache) and replaces its 1,000-class output layer, called `fc` ("fully connected"), with a new 8-class layer.

In [ ]:
def build_resnet18():
    """ResNet-18 with pretrained ImageNet weights and a new 8-class output layer."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)    # ResNet-18 with the best available ImageNet weights
    model.fc = nn.Linear(512, len(CLASSES))                             # The last layer gets 512 features per image; new layer: 512 -> 8 class scores (random starting weights)
    return model

`set_backbone_trainable` freezes or unfreezes the pretrained part of the network (the "backbone", every layer except `fc`). A weight with `requires_grad = False` gets no gradient in `loss.backward()`, so `optimiser.step()` does not change it.

In [ ]:
def set_backbone_trainable(model, trainable):
    """Freeze (trainable=False) or unfreeze (trainable=True) every layer except the fc output layer."""
    for name, parameter in model.named_parameters():                    # name is e.g. "layer1.0.conv1.weight" or "fc.weight"
        if not name.startswith("fc"):                                   # Skip the output layer, which is always trained
            parameter.requires_grad = trainable                         # False = frozen (no gradient, no update); True = trained

`count_trainable_weights` counts the weights that will be updated during training, to check the freezing worked.

In [ ]:
def count_trainable_weights(model):
    """Return the number of weights with requires_grad = True."""
    trainable_weights = 0
    for parameter in model.parameters():
        if parameter.requires_grad:                                     # Only weights that get gradients are trained
            trainable_weights += parameter.numel()                      # numel() = number of values in the tensor
    return trainable_weights

## ResNet-18 training (section 3.9 of kaggle_comp.ipynb)

The cells below are section 3.9 of the main notebook, without its `if RUNNING_ON_KAGGLE:` guards, because running them is the purpose of this notebook. The only changes are that the loaders use `NUM_WORKERS` (4 on Kaggle, 0 on the laptop) instead of `KAGGLE_NUM_WORKERS`, so the laptop smoke test works, and that the output file names use `resnet-18`.

ResNet-18 uses `RESNET_IMAGE_SIZE` x `RESNET_IMAGE_SIZE` images (128 or 224), so the transforms and loaders are built at that size. The steps are the same as Task 2 (and the same ImageNet normalisation); only the size changes. When `USE_AUGMENTATION` is `False`, the training loader uses the evaluation transform instead (resize and normalise only), so training images are not augmented.

In [ ]:
# Training images: the same augmentation as Task 2, at RESNET_IMAGE_SIZE x RESNET_IMAGE_SIZE
resnet_training_transform = transforms.Compose([
    transforms.RandomResizedCrop(RESNET_IMAGE_SIZE, scale=(0.8, 1.0)),     # Random 80-100% crop, resized to RESNET_IMAGE_SIZE
    transforms.RandomHorizontalFlip(),                                      # Mirror left-right half of the time
    transforms.RandomRotation(15),                                          # Rotate by up to 15 degrees either way
    transforms.ColorJitter(brightness=0.2, contrast=0.2),                   # Change brightness and contrast by up to 20%
    transforms.RandomGrayscale(p=0.1),                                      # Make 10% of images greyscale (stops "greyscale means cat")
    transforms.ToTensor(),                                                  # PIL image -> 3 x size x size tensor, values 0-1
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),             # Same ImageNet normalisation as every other model
])

# Validation and test images: resize and normalise only
resnet_evaluation_transform = transforms.Compose([
    transforms.Resize((RESNET_IMAGE_SIZE, RESNET_IMAGE_SIZE)),              # RESNET_IMAGE_SIZE x RESNET_IMAGE_SIZE
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Training images: augmented, unless USE_AUGMENTATION is False (augmentation experiment)
if USE_AUGMENTATION:
    train_transform_used = resnet_training_transform                        # Random crop, flip, rotation, colour changes and greyscale
else:
    train_transform_used = resnet_evaluation_transform                      # No augmentation: resize and normalise only

resnet_train_loader = DataLoader(FacialFeatureDataset(df_train_split, train_transform_used),
                                 batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)       # Shuffled every epoch
resnet_validation_loader = DataLoader(FacialFeatureDataset(df_validation_split, resnet_evaluation_transform),
                                      batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
resnet_test_loader = DataLoader(TestImageDataset(df_test_info, resnet_evaluation_transform),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)       # Same order as the image IDs

print("Device:", DEVICE)
print("Training batches:  ", len(resnet_train_loader))
print("Validation batches:", len(resnet_validation_loader))
print("Test batches:      ", len(resnet_test_loader))

**Stage 1: frozen backbone.** Only the new 8-class `fc` layer is trained, for `RESNET_FROZEN_EPOCHS` epochs at `RESNET_FROZEN_LEARNING_RATE`. The frozen weights get no gradients, so Adam leaves them unchanged. (Batch norm layers still update their running averages from these images, which helps the network adjust to them.)

Each stage calls `train_model` separately, and `train_model` starts its "best so far" from 0. So stage 1 saves to `resnet-18_frozen_best.pt` and stage 2 to `resnet-18_best.pt`; otherwise the first fine-tuning epoch would always overwrite stage 1's best weights, even if it were worse. The better of the two files is used for the submission.

In [ ]:
torch.manual_seed(RANDOM_SEED)                                                  # Same new fc weights, shuffling and augmentation every run
resnet_model = build_resnet18().to(DEVICE)                                      # Pretrained ResNet-18 with a new 8-class layer
set_backbone_trainable(resnet_model, False)                                     # Freeze everything except fc
print(f"Stage 1 trainable weights: {count_trainable_weights(resnet_model):,}")

frozen_optimiser = torch.optim.Adam(resnet_model.parameters(), lr=RESNET_FROZEN_LEARNING_RATE)   # Frozen weights have no gradients, so only fc changes
df_resnet_frozen_history = train_model(resnet_model, resnet_train_loader, resnet_validation_loader,
                                       loss_function, frozen_optimiser, RESNET_FROZEN_EPOCHS, model_name="resnet-18_frozen")

**Stage 2: fine-tuning.** The backbone is unfrozen and the whole network is trained for `RESNET_FINE_TUNE_EPOCHS` more epochs, starting from the end of stage 1. A new optimiser is created, with the 10 times smaller `RESNET_FINE_TUNE_LEARNING_RATE`, so the pretrained features are adjusted gently instead of being overwritten.

In [ ]:
set_backbone_trainable(resnet_model, True)                                      # Unfreeze every layer
print(f"Stage 2 trainable weights: {count_trainable_weights(resnet_model):,}")

fine_tune_optimiser = torch.optim.Adam(resnet_model.parameters(), lr=RESNET_FINE_TUNE_LEARNING_RATE)   # New optimiser for all 11 million weights
df_resnet_fine_tune_history = train_model(resnet_model, resnet_train_loader, resnet_validation_loader,
                                          loss_function, fine_tune_optimiser, RESNET_FINE_TUNE_EPOCHS, model_name="resnet-18")

Join the two stages into one history (fine-tuning epochs are renumbered to follow on from stage 1), save it, and find the best epoch and the weights file it was saved in.

In [ ]:
df_resnet_frozen_history["stage"] = "frozen"                                    # Label which stage each epoch belongs to
df_resnet_fine_tune_history["stage"] = "fine-tune"
df_resnet_fine_tune_history["epoch"] = df_resnet_fine_tune_history["epoch"] + RESNET_FROZEN_EPOCHS   # 1-7 -> 4-10
df_resnet_history = pd.concat([df_resnet_frozen_history, df_resnet_fine_tune_history], ignore_index=True)

df_resnet_history.to_csv(models_folder / "resnet-18_history.csv", index=False)  # Next to the model weights
df_resnet_history.to_csv(figures_folder / "resnet-18_history.csv", index=False) # Small copy for the report and git
print(df_resnet_history)

# Best epoch over both stages, and the file its weights were saved in
best_resnet_index = df_resnet_history["validation_accuracy"].idxmax()
best_resnet_epoch = int(df_resnet_history.loc[best_resnet_index, "epoch"])
best_resnet_accuracy = df_resnet_history.loc[best_resnet_index, "validation_accuracy"]
best_resnet_stage = df_resnet_history.loc[best_resnet_index, "stage"]
if best_resnet_stage == "frozen":
    resnet_weights_path = models_folder / "resnet-18_frozen_best.pt"
else:
    resnet_weights_path = models_folder / "resnet-18_best.pt"

print()
print(f"Best validation accuracy: {best_resnet_accuracy * 100:.2f}% at epoch {best_resnet_epoch} ({best_resnet_stage} stage)")
print("Best weights:", resnet_weights_path)
print(f"Total training time: {df_resnet_history['minutes'].sum():.0f} minutes")

ResNet-18 learning curves: loss (left) and accuracy (right). The dotted line marks where the backbone was unfrozen, and the dashed line the best epoch.

In [ ]:
resnet_epochs = df_resnet_history["epoch"]                                      # 1 to 10, for the x axis
unfreeze_position = RESNET_FROZEN_EPOCHS + 0.5                                  # Between the last frozen epoch and the first fine-tuning epoch

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 4))

# Left: loss per epoch (circles = training, squares = validation)
ax_left.plot(resnet_epochs, df_resnet_history["train_loss"], marker="o", label="Training")
ax_left.plot(resnet_epochs, df_resnet_history["validation_loss"], marker="s", label="Validation")
ax_left.axvline(unfreeze_position, color="grey", linestyle=":", linewidth=1, label="Backbone unfrozen")
ax_left.set_xlabel("Epoch")
ax_left.set_ylabel("Loss (cross-entropy)")
ax_left.set_title("ResNet-18 loss")
ax_left.set_xticks(resnet_epochs)
ax_left.legend(frameon=False)

# Right: accuracy per epoch
ax_right.plot(resnet_epochs, df_resnet_history["train_accuracy"] * 100, marker="o", label="Training")
ax_right.plot(resnet_epochs, df_resnet_history["validation_accuracy"] * 100, marker="s", label="Validation")
ax_right.axvline(unfreeze_position, color="grey", linestyle=":", linewidth=1, label="Backbone unfrozen")
ax_right.axvline(best_resnet_epoch, color="grey", linestyle="--", linewidth=1, label=f"Best epoch ({best_resnet_epoch})")
ax_right.set_xlabel("Epoch")
ax_right.set_ylabel("Accuracy (%)")
ax_right.set_title(f"Best validation accuracy: {best_resnet_accuracy * 100:.1f}% at epoch {best_resnet_epoch}")
ax_right.set_xticks(resnet_epochs)
ax_right.legend(frameon=False)

plt.suptitle(f"ResNet-18 learning curves ({RESNET_IMAGE_SIZE} x {RESNET_IMAGE_SIZE} images, {RESNET_FROZEN_EPOCHS} frozen + {RESNET_FINE_TUNE_EPOCHS} fine-tuning epochs)")
plt.tight_layout()                                                              # Stop the two plots overlapping
plt.savefig(figures_folder / "resnet-18_learning_curves.png", dpi=150, bbox_inches="tight")   # Save the figure for the report
plt.show()

Load the best ResNet-18 weights, predict every test image, and save the submission file in the same format as `sample_submission.csv`.

In [ ]:
best_resnet_model = build_resnet18().to(DEVICE)                                 # Same layers (the pretrained weights are replaced next)
best_resnet_model.load_state_dict(torch.load(resnet_weights_path, map_location=DEVICE, weights_only=True))   # Copy in the best saved weights
print("Loaded weights from", resnet_weights_path)

resnet_test_ids, resnet_test_predictions = predict(best_resnet_model, resnet_test_loader)   # One class number per test image
print("Predicted", len(resnet_test_predictions), "test images")

# Turn each class number into its class name
resnet_class_names = []
for class_number in resnet_test_predictions:
    resnet_class_names.append(index_to_class[class_number])

df_submission_resnet18 = pd.DataFrame({"image_id": resnet_test_ids, "class": resnet_class_names})
submission_resnet18_path = submissions_folder / "submission_resnet-18.csv"
df_submission_resnet18.to_csv(submission_resnet18_path, index=False)           # index=False: no extra row-number column
print("Saved", submission_resnet18_path)
print()
print(df_submission_resnet18.head())

The same checks as the baseline submission (section 3.7), on the saved ResNet-18 file. In a smoke test only 640 test images are predicted, so the 29,443-row check is skipped and the image-ID check only confirms that every ID is a real test image.

In [ ]:
df_check = pd.read_csv(submission_resnet18_path)                                # Read the saved file back, to check exactly what Kaggle will receive

if SMOKE_TEST:
    print(f"SKIPPED (smoke test): {len(df_check)} rows, the full run has 29,443")
else:
    print_check(f"{len(df_check)} rows (should be 29,443)", len(df_check) == 29443)

submission_ids = set(df_check["image_id"])
sample_ids = set(df_sample_submission["image_id"])
if SMOKE_TEST:
    print_check("every image ID is in sample_submission.csv (smoke test)", submission_ids.issubset(sample_ids))
else:
    print_check("same set of image IDs as sample_submission.csv", submission_ids == sample_ids)

all_classes_valid = True
for class_name in df_check["class"]:
    if class_name not in CLASSES:
        all_classes_valid = False
print_check("every class name is in CLASSES", all_classes_valid)

print_check(f"columns {list(df_check.columns)} match sample_submission.csv", list(df_check.columns) == list(df_sample_submission.columns))

### Results and files to download

The best epoch and the training time, then every file this run wrote. On Kaggle these are in the Output tab.

In [ ]:
print(f"Image size:               {RESNET_IMAGE_SIZE} x {RESNET_IMAGE_SIZE}")
print(f"Augmentation:             {USE_AUGMENTATION}")
print(f"Best validation accuracy: {best_resnet_accuracy * 100:.2f}% at epoch {best_resnet_epoch} ({best_resnet_stage} stage)")
print(f"Total training time:      {df_resnet_history['minutes'].sum():.1f} minutes")
print(f"Average time per epoch:   {df_resnet_history['minutes'].mean():.2f} minutes")
print("Smoke test:              ", SMOKE_TEST)
print()

print("Files written:")
for folder in [figures_folder, models_folder, submissions_folder]:
    for file_path in sorted(folder.iterdir()):
        size_mb = file_path.stat().st_size / 1024 / 1024                        # Bytes -> megabytes
        print(f"  {str(file_path):<62} {size_mb:8.2f} MB")